# Import Library

In [1]:
import os
import requests
import tarfile
import time
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split ,Subset
import torchvision.models as models
import torch.nn as nn
import torch
import PIL.Image
import pathlib
from torchsummary import summary
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import torch.optim as optim
import pandas as pd
from torchvision.transforms import functional as TF
import timm
from transformers import AutoImageProcessor, AutoModelForImageClassification
import torchvision.transforms as transforms
import torchvision.transforms.v2 as v2
from torchvision.transforms import AutoAugment, AutoAugmentPolicy

from torch.cuda.amp import autocast, GradScaler # Mixed Precision

if torch.cuda.is_available():
    device = "cuda" # Use NVIDIA GPU (if available)
elif torch.backends.mps.is_available():
    device = "mps" # Use Apple Silicon GPU (if available)
else:
    device = "cpu" # Default to CPU if no GPU is available

class_names = ['birds', 'bottles', 'breads', 'butterflies', 'cakes', 'cats', 'chickens', 'cows', 'dogs', 'ducks',
                  'elephants', 'fishes', 'handguns', 'horses', 'lions', 'lipsticks', 'seals', 'snakes', 'spiders', 'vases']


/home/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Dataset & DataLoader Function

In [2]:
def construct_data(random_seed , processor , train_percent ,random_augment_ops):
    torch.manual_seed(random_seed)

    # Training Data Augmentation
    train_transform = transforms.Compose([
            # Resize
            transforms.Resize(256),
            # Random Crop 
            transforms.RandomResizedCrop(size = 224 , scale =(0.08,1) , ratio =(3/4 , 4/3)),
            # Random Augment
            transforms.RandAugment(interpolation=transforms.InterpolationMode.BILINEAR , num_ops = random_augment_ops),
            # Auto Augment
            transforms.AutoAugment(policy = AutoAugmentPolicy.IMAGENET),
            # Horizontal Flip
            transforms.RandomHorizontalFlip(p=0.5),
            # Color Jitter
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1 , hue = 0.02),
            # Convert to Tensor
            transforms.Lambda(lambda img: processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)),
            #transforms.RandomErasing(p=0.1),
                                          ])
    # Validation Data Augmentation
    val_transform = transforms.Compose([
            # Resize
            transforms.Resize(256),
            transforms.Lambda(lambda img: processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)),
    ])
    
    
    # Load the dataset using torchvision.datasets.ImageFolder and apply transformations
    data_dir = "./FIT5215_Dataset"
    full_dataset = datasets.ImageFolder(data_dir)
    
    # Split
    train_size = int(train_percent * len(full_dataset))
    val_size   = len(full_dataset) - train_size
    train_subset, val_subset = random_split(full_dataset, [train_size, val_size])
    
    # Re-wrap with transforms
    train_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
    val_dataset   = datasets.ImageFolder(data_dir, transform=val_transform)
    
    # Apply same indices from split
    train_dataset = Subset(train_dataset, train_subset.indices)
    val_dataset   = Subset(val_dataset, val_subset.indices)
    
    # DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                              num_workers=4)
    val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False,
                              num_workers=4)

    transform_test_Data = transforms.Compose([
         # Resize
            transforms.Resize(256),
            transforms.Lambda(lambda img: processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0))])


    test_dir = "./test_set"
    test_dataset = datasets.ImageFolder(test_dir , transform = transform_test_Data)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    return train_loader , val_loader , test_loader
    

# Training Function

In [4]:
import numpy as np

def top_k_accuracy(output, target, k=5):
    batch_size = target.size(0)
    _, pred = output.topk(k, 1, True, True)  # Get top-k predictions
    pred = pred.t()  # Transpose predictions for comparison
    correct = pred.eq(target.reshape(1, -1).expand_as(pred))  # Compare predictions with target
    correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)  # Calculate correct top-k
    return correct_k.mul_(1.0 / batch_size).item()  # Calculate top-k accuracy

class BaseTrainer:
    def __init__(self, model, criterion, optimizer, train_loader, val_loader ,device,random_seed,train_percent ,random_augment_ops,rand_epoch):
        self.model = model.to(device)
        self.criterion = criterion
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.stop_training = False
        self.random_seed = random_seed
        self.train_percent = train_percent
        self.random_augment_ops = random_augment_ops
        self.rand_epoch = rand_epoch

        # Mixed Precision Training
        self.scaler = GradScaler()
    def fit(self, num_epochs):
        self.num_batches = len(self.train_loader)
        best_val_loss = float('inf')
        for epoch in range(num_epochs):
            if self.stop_training:
                break
            print(f'Epoch {epoch + 1}/{num_epochs}')
            start_time = time.time()
            train_loss, train_accuracy = self.train_one_epoch()
            val_loss, top1_acc, top5_acc = self.validate_one_epoch()
            print(
                f' train_loss: {train_loss:.4f} - train_accuracy: {train_accuracy:.4f} - val_loss: {val_loss:.4f} - top1_acc: {top1_acc:.4f} - top5_acc: {top5_acc:.4f}'
            )
            self.on_epoch_end({'epoch': epoch, 'train_loss': train_loss, 'train_accuracy': train_accuracy, 'val_loss': val_loss, 'val_accuracy': top1_acc, 'top5_acc': top5_acc})

            if val_loss <  best_val_loss:
                save_path = f"models/Register_Dino_Random_Giant_seed{self.random_seed}_train{self.train_percent}_aug{self.random_augment_ops}_epoch{self.rand_epoch}.pth"
                torch.save(self.model.state_dict(), save_path)


    def train_one_epoch(self):
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0

        for i, data in enumerate(self.train_loader):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            ##### Probability Getting Cutmix End
            self.optimizer.zero_grad()

            # Mixed Precision Training
            with autocast( dtype=torch.float16):
                outputs = self.model(inputs)
                outputs = outputs.logits
                loss = self.criterion(outputs, labels)
            
            
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            
            #loss.backward()
            #self.optimizer.step()
            self.scaler.update()

            running_loss += loss.item()

            # Accuracy Calculation Cutmix
            if labels.ndim == 2:
                hard_label = torch.argmax(labels,dim=1)
            else:
                hard_label = labels
            _, predicted = torch.max(outputs.data, 1)
            total += hard_label.size(0)
            correct += (predicted == hard_label).sum().item()

        train_accuracy = correct / total
        train_loss = running_loss / self.num_batches
        return train_loss, train_accuracy

    def validate_one_epoch(self):
        self.model.eval()
        val_loss, correct, total_top1_acc, total_top5_acc, total = 0.0, 0, 0.0, 0.0, 0
        with torch.no_grad():
            for data in self.val_loader:
                inputs, labels = data
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = self.model(inputs)
                outputs = outputs.logits
                loss = self.criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                correct += (predicted == labels).sum().item()

                top1_acc = top_k_accuracy(outputs, labels, k=1)
                top5_acc = top_k_accuracy(outputs, labels, k=5)

                total_top1_acc += top1_acc * inputs.size(0)  # Custom evaluation metric
                total_top5_acc += top5_acc * inputs.size(0)  # Custom evaluation metric
                total += labels.size(0)

        val_loss /= len(self.val_loader)
        top1_acc = total_top1_acc / total
        top5_acc = total_top5_acc / total
        return val_loss, top1_acc, top5_acc

    def on_epoch_end(self, params):
        pass

# Save To Csv

In [5]:
from tqdm import tqdm
def save_prediction_to_csv(model, loader, device, output_file="SubmissionDefault.csv"):
    model.eval()
    predictions = []
    image_ids = []
    df = {
    "ID": [],
    "Label": []
    }
    total = 0
    with torch.no_grad():
        for i, (batchX, batchY) in tqdm(enumerate(loader)):
            batchX, batchY = batchX.to(device), batchY.to(device)
            outputs = model(batchX.float())  # Convert to float32 and feed batch to the model
            outputs = outputs.logits
            predicted = torch.argmax(outputs, dim=1)  # Get the predicted class
            total += predicted.size(0)
            for ids, pred in enumerate(predicted):
                label = class_names[pred.to(device).item()]
                df["ID"].append(i*500+ids)
                df["Label"].append(label)
    df["ID"] = [i for i in range(total)]
    # Create a DataFrame
    df = pd.DataFrame(df)
    # Save to CSV
    df.to_csv(output_file, index=False)

# Automation

In [6]:
import numpy as np
[np.random.randint(1,100000) for i in range (13)]

[37098,
 2409,
 81202,
 47578,
 41095,
 52130,
 22093,
 84567,
 84170,
 65236,
 96808,
 95136,
 12490]

In [7]:

if torch.cuda.is_available():
    device = "cuda" # Use NVIDIA GPU (if available)
elif torch.backends.mps.is_available():
    device = "mps" # Use Apple Silicon GPU (if available)
else:
    device = "cpu" # Default to CPU if no GPU is available

    
seed_list = [np.random.randint(1,100000) for i in range (5)]
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-with-registers-giant-imagenet1k-1-layer')
for idx,seed in enumerate(seed_list):
    print('=======================================')
    train_percent = np.random.choice([0.7,0.8]).item()
    random_augment_ops = np.random.choice([1,2]).item()
    epoch_num = [30,20,35,20,25,23,23,22,24,23][idx]#np.random.choice([20,22,23,25]).item()
    # Construct Data Loader
    train_loader , val_loader , test_loader = construct_data(  random_seed = seed , 
                                                                processor = processor , 
                                                                train_percent = train_percent ,
                                                                random_augment_ops = random_augment_ops)
    # Make The Model
    model = AutoModelForImageClassification.from_pretrained('facebook/dinov2-with-registers-giant-imagenet1k-1-layer')
    model.classifier = nn.Linear(in_features=3072, out_features=20, bias=True)
    model.to(device)
    criterion = nn.CrossEntropyLoss()

    # Define Optimizer
    param_list = []
    for idx ,(name, param) in enumerate(model.named_parameters()):
        if idx < 690:
            param.requires_grad = False

        else:
            param.requires_grad = True
            param_list.append(param)

    optimizer = optim.SGD(param_list, lr=0.001, momentum=0.9, weight_decay =1e-4 , nesterov = True)

    # Train The Model
    trainer = BaseTrainer(model, criterion, optimizer, train_loader, val_loader ,device,random_seed = seed,train_percent = train_percent ,random_augment_ops = random_augment_ops,rand_epoch=epoch_num)
    trainer.fit(num_epochs = epoch_num)

    # Load The Best Model
    model = AutoModelForImageClassification.from_pretrained('facebook/dinov2-with-registers-giant-imagenet1k-1-layer')
    model.classifier = nn.Linear(in_features=3072, out_features=20, bias=True)
    path = f"./models/Register_Dino_Random_Giant_seed{seed}_train{train_percent}_aug{random_augment_ops}_epoch{epoch_num}.pth"
    model.load_state_dict(torch.load(path))
    model.to(device)

    # Save to Csv
    filename = f"Register_Dino_Random_Giant_seed{seed}_train{train_percent}_aug{random_augment_ops}_epoch{epoch_num}.csv"
    save_prediction_to_csv(model, test_loader, device, output_file=filename)
    df = pd.read_csv(filename)
    df

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 7332.70it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Epoch 1/30


/tmp/ipykernel_86/1346544270.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler()
/tmp/ipykernel_86/1346544270.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast( dtype=torch.float16):


 train_loss: 0.2941 - train_accuracy: 0.9284 - val_loss: 0.0272 - top1_acc: 0.9968 - top5_acc: 0.9995
Epoch 2/30
 train_loss: 0.1156 - train_accuracy: 0.9684 - val_loss: 0.0234 - top1_acc: 0.9968 - top5_acc: 1.0000
Epoch 3/30
 train_loss: 0.0970 - train_accuracy: 0.9720 - val_loss: 0.0239 - top1_acc: 0.9958 - top5_acc: 1.0000
Epoch 4/30
 train_loss: 0.0907 - train_accuracy: 0.9745 - val_loss: 0.0226 - top1_acc: 0.9968 - top5_acc: 0.9995
Epoch 5/30
 train_loss: 0.0871 - train_accuracy: 0.9746 - val_loss: 0.0239 - top1_acc: 0.9968 - top5_acc: 0.9995
Epoch 6/30
 train_loss: 0.0847 - train_accuracy: 0.9749 - val_loss: 0.0237 - top1_acc: 0.9958 - top5_acc: 0.9995
Epoch 7/30
 train_loss: 0.0793 - train_accuracy: 0.9757 - val_loss: 0.0228 - top1_acc: 0.9963 - top5_acc: 1.0000
Epoch 8/30
 train_loss: 0.0825 - train_accuracy: 0.9750 - val_loss: 0.0228 - top1_acc: 0.9963 - top5_acc: 0.9995
Epoch 9/30


KeyboardInterrupt: 

In [ ]:
print('test')


In [ ]:
torch.cuda.empty_cache()

# Classification Report

In [2]:
import os
import requests
import tarfile
import time
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split ,Subset
import torchvision.models as models
import torch.nn as nn
import torch
import PIL.Image
import pathlib
from torchsummary import summary
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import torch.optim as optim
import pandas as pd
from torchvision.transforms import functional as TF
import timm
from transformers import AutoImageProcessor, AutoModelForImageClassification
import torchvision.transforms as transforms
import torchvision.transforms.v2 as v2
from torchvision.transforms import AutoAugment, AutoAugmentPolicy
from sklearn.metrics import classification_report
from torch.cuda.amp import autocast, GradScaler # Mixed Precision
from sklearn.metrics import classification_report

if torch.cuda.is_available():
    device = "cuda" # Use NVIDIA GPU (if available)
elif torch.backends.mps.is_available():
    device = "mps" # Use Apple Silicon GPU (if available)
else:
    device = "cpu" # Default to CPU if no GPU is available

class_names = ['birds', 'bottles', 'breads', 'butterflies', 'cakes', 'cats', 'chickens', 'cows', 'dogs', 'ducks',
                  'elephants', 'fishes', 'handguns', 'horses', 'lions', 'lipsticks', 'seals', 'snakes', 'spiders', 'vases']


/home/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

processor = AutoImageProcessor.from_pretrained('facebook/dinov2-with-registers-giant-imagenet1k-1-layer')

transform_test_Data = transforms.Compose([
     # Resize
        transforms.Resize(256),
        transforms.Lambda(lambda img: processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0))])


# Load the dataset using torchvision.datasets.ImageFolder and apply transformations
data_dir = "./FIT5215_Dataset"
full_dataset = datasets.ImageFolder(data_dir, transform = transform_test_Data)



# DataLoaders
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True,
                          num_workers=4, pin_memory=True)




    

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 13751.82it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
model = AutoModelForImageClassification.from_pretrained('facebook/dinov2-with-registers-giant-imagenet1k-1-layer')
model.classifier = nn.Linear(in_features=3072, out_features=20, bias=True)
model.load_state_dict(torch.load(f"./models/Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.pth"))

<All keys matched successfully>

In [5]:
model.to(device)
for i, data in enumerate(full_loader):
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)
    outputs = model(inputs)
    outputs_logits = outputs.logits
    break

    

OutOfMemoryError: CUDA out of memory. Tried to allocate 132.00 MiB. GPU 0 has a total capacity of 23.56 GiB of which 78.88 MiB is free. Including non-PyTorch memory, this process has 23.46 GiB memory in use. Of the allocated memory 22.81 GiB is allocated by PyTorch, and 418.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [6]:
!nvidia-smi


Sat Sep 20 02:22:26 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.06             Driver Version: 570.124.06     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               Off |   00000000:41:00.0 Off |                  Off |
| 30%   33C    P8             22W /  230W |   24045MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----